# Build Data Pipelines with Lakeflow Spark Declarative Pipelines

Training notebook covering Lakeflow Spark Declarative Pipelines (SDP): streaming tables, materialized views, expectations, Auto Loader integration, medallion architecture, and pipeline operations.

## 📋 What We'll Cover

1. **What is Lakeflow Spark Declarative Pipelines** — purpose, how it differs from regular Spark jobs
2. **Key Concepts** — streaming tables, materialized views, temporary views, expectations
3. **Python API** — `import dlt` and the newer `from pyspark import pipelines as dp`
4. **Medallion Architecture in SDP** — bronze → silver → gold pattern
5. **Auto Loader Integration** — incrementally ingesting files with `cloudFiles`
6. **Data Quality with Expectations** — `@dlt.expect`, `expect_or_drop`, `expect_or_fail`
7. **This Training's Pipeline** — config-driven pipeline files and what they each do
8. **Creating & Running a Pipeline** — UI setup, configuration, and triggering runs
9. **Monitoring** — event logs, pipeline graph, data quality metrics
10. **@dp vs @dlt — API Comparison** — when to use the modern vs classic API, side-by-side examples

## 1. What is Lakeflow Spark Declarative Pipelines (SDP)

Lakeflow Spark Declarative Pipelines is Databricks' **declarative ETL framework**. Instead of writing imperative Spark code that manually manages reads, writes, checkpoints, and retries, you **declare what tables you want** and SDP figures out how to build and keep them up to date.

### SDP vs Regular Spark Job

| Aspect | Regular Spark Job | Lakeflow SDP |
|---|---|---|
| **Write mode** | You manage `overwrite` / `append` / `merge` | Declared automatically from the function type |
| **Checkpoints** | You create and manage checkpoint paths | Managed by SDP automatically |
| **Dependencies** | You handle ordering manually | SDP builds the DAG from `LIVE.table` references |
| **Data quality** | Custom `filter` / `assert` code | Built-in expectations with metrics tracking |
| **Retries on failure** | Job retry policy only | SDP retries at the table level |
| **Lineage** | Not automatic | Auto-tracked in Unity Catalog |

### How SDP works

```
You write Python functions ──────────────► SDP builds an execution DAG
                                            │
   @dlt.table                               ▼
   def bronze():                    Streaming Table (incremental)
       return readStream...         Materialized View (full recompute)
                                            │
   @dlt.table                               ▼
   def silver():                    Delta table in Unity Catalog
       return dlt.read("bronze")...
```

## 2. Key Concepts

### Streaming Table
A table populated **incrementally** — each pipeline run processes only new data. Backed by Structured Streaming. Use for:
* Raw ingestion from files, Kafka, CDC
* Any source where you only want to process new records

```python
import dlt

@dlt.table
def my_streaming_table():
    return spark.readStream.format("cloudFiles").load("/path/to/files")
```

### Materialized View
A table that is **fully recomputed** on each run. Use for:
* Aggregations, joins, and lookups where the full result set must be consistent
* Gold-layer reporting tables

```python
@dlt.table
def my_materialized_view():
    return spark.read.table("LIVE.my_streaming_table").groupBy("key").count()
```

### Temporary View
A virtual table visible only within the same pipeline run. Not persisted to Unity Catalog. Use for intermediate transformations you don't need to expose.

```python
@dlt.view
def my_temp_view():
    return dlt.read("my_streaming_table").filter(col("status") == "active")
```

### Key rule: reference pipeline tables with `LIVE.` or `dlt.read()`
When reading a table defined **within the same pipeline**, use:
* `spark.readStream.table("LIVE.table_name")` for streaming reads
* `spark.read.table("LIVE.table_name")` or `dlt.read("table_name")` for batch reads

## 3. Python API

Databricks supports two equivalent Python APIs. Both work today; the newer `pyspark.pipelines` API aligns with the open-source Apache Spark Declarative Pipelines standard.

### Classic API (`import dlt`)
```python
import dlt
from pyspark.sql.functions import col

@dlt.table(comment="Bronze customer data")
def customer_bronze():
    return spark.readStream.format("cloudFiles").load("/path")

@dlt.expect_or_drop("valid_key", "c_custkey IS NOT NULL")
@dlt.table(comment="Cleaned customer data")
def customer_silver():
    return dlt.read_stream("customer_bronze").filter(col("c_acctbal") >= 0)
```

### Newer API (`from pyspark import pipelines as dp`)
```python
from pyspark import pipelines as dp
from pyspark.sql.functions import col

@dp.table(comment="Bronze customer data")
def customer_bronze():
    return spark.readStream.format("cloudFiles").load("/path")

@dp.expect_or_drop("valid_key", "c_custkey IS NOT NULL")
@dp.table(comment="Cleaned customer data")
def customer_silver():
    return spark.readStream.table("customer_bronze").filter(col("c_acctbal") >= 0)
```

### Decorator name mapping

| Classic | Newer (pyspark.pipelines) | Creates |
|---|---|---|
| `@dlt.table` + `readStream` | `@dp.table` + `readStream` | Streaming table |
| `@dlt.table` + `read` | `@dp.materialized_view` + `read` | Materialized view |
| `@dlt.view` | `@dp.temporary_view` | Temporary view |
| `@dlt.expect` | `@dp.expect` | Expectation (warn) |
| `@dlt.expect_or_drop` | `@dp.expect_or_drop` | Expectation (drop row) |
| `@dlt.expect_or_fail` | `@dp.expect_or_fail` | Expectation (fail pipeline) |

> 💡 This training uses `import dlt` (classic API) as it is most widely documented today.

## 4. Medallion Architecture in SDP

SDP is designed around the medallion (bronze/silver/gold) pattern:

```
  Source                Bronze                  Silver                  Gold
 (CSV files)       (raw ingestion)         (cleaned/enriched)     (aggregated)

 abfss://...  ──►  customer_bronze  ──►  customer_silver  ──►  customer_gold
  (Auto Loader)    Streaming Table      Streaming Table        Materialized View
                   • raw schema         • typed columns         • aggregations
                   • all rows           • nulls dropped         • business metrics
                   • _rescued_data      • enriched fields       • dashboard ready
```

### Why each layer matters

| Layer | Purpose | SDP type |
|---|---|---|
| **Bronze** | Exact copy of source data; never transform, never drop rows | Streaming table (Auto Loader) |
| **Silver** | Clean, type-correct, enriched; expectations enforced | Streaming table |
| **Gold** | Business aggregations and KPIs; queryable by BI tools | Materialized view |

### Pipeline dependency graph
SDP automatically resolves the execution order from your `LIVE.table` references:
```
Auto Loader scan ─► customer_bronze ─► customer_silver ─► customer_gold
```
You never write `depends_on` manually — SDP infers it.

## 5. Auto Loader Integration

Auto Loader (`cloudFiles` format) is the standard way to ingest files from cloud storage into a bronze streaming table. SDP manages the checkpoint automatically.

### How Auto Loader tracks files

```
First run:  reads all existing files ──► writes to bronze table
                                          ↓ records "last seen" file list
Second run: reads ONLY new files ─────► appends to bronze table
```

### Key cloudFiles options

| Option | Description |
|---|---|
| `cloudFiles.format` | File format: `csv`, `json`, `parquet`, `avro` |
| `cloudFiles.inferColumnTypes` | Auto-detect column types (vs treating all as string) |
| `cloudFiles.schemaEvolutionMode` | How to handle new columns: `addNewColumns` (default), `rescue`, `failOnNewColumns` |
| `header` | For CSV: whether the first row is a header |
| `cloudFiles.schemaHints` | Hint types for specific columns, e.g. `c_acctbal double, c_custkey int` |

### Schema rescue
When Auto Loader encounters a column it didn't see before, it puts the raw row into a `_rescued_data` column rather than failing. Set an expectation to alert on unexpected rescues:
```python
@dlt.expect("no rescued data", "_rescued_data IS NULL")
```

## 6. Data Quality with Expectations

Expectations are **declarative data quality rules** attached to a table. SDP evaluates them on every row and records pass/fail metrics in the event log.

### Three enforcement modes

| Decorator | Row fails expectation → | Use when |
|---|---|---|
| `@dlt.expect(name, expr)` | Row is kept, failure is **logged** (warn) | You want to track quality without blocking |
| `@dlt.expect_or_drop(name, expr)` | Row is **dropped** from the table | Invalid rows must not pollute downstream |
| `@dlt.expect_or_fail(name, expr)` | Entire **pipeline fails** | Zero-tolerance; data must be perfect |

### Examples

```python
@dlt.expect("no rescued data", "_rescued_data IS NULL")          # warn only
@dlt.expect_or_drop("valid_key",  "c_custkey IS NOT NULL")        # drop bad rows
@dlt.expect_or_drop("positive_balance", "c_acctbal >= -999")      # business rule
@dlt.expect_or_fail("valid_segment",  "c_mktsegment IS NOT NULL") # must never be null
@dlt.table(comment="Silver: cleaned customer data")
def customer_silver():
    return dlt.read_stream("customer_bronze")
```

### Where to see expectation metrics
In the Pipeline UI → click a table node → **Data quality** tab shows pass/fail counts per expectation per run.

## 7. This Training's Pipeline

This training uses a **configuration-driven** approach: all tuneable parameters live in a JSON config file, and a single Python file defines both bronze and silver tables for **two datasets** (customer + orders).

We provide **two versions** of the pipeline file so you can compare the classic `@dlt` API with the modern `@dp` API side by side.

### `sdp_pipeline_config.json` — shared configuration
All source paths, schemas, table names, expectations, and silver transforms are defined here — **not hardcoded in Python**.
* **Customer** tables: `customer_bronze` → `customer_silver` (CSV from `abfss://inboundfiles@unityms.dfs.core.windows.net/customer`)
* **Orders** tables: `orders_bronze` → `orders_silver` (CSV from `abfss://inboundfiles@unityms.dfs.core.windows.net/orders`)

### `sdp_customer_pipeline.py` — classic `@dlt` API version
* Uses `import dlt` and `@dlt.table()`, `@dlt.expect()`
* Reads pipeline tables with `dlt.read_stream("table_name")`
* Good baseline if you work with existing DLT pipelines or older documentation

### `sdp_customer_pipeline_dp.py` — modern `@dp` API version
* Uses `from pyspark import pipelines as dp` and `@dp.table()`, `@dp.expect()`
* Reads pipeline tables with `spark.readStream.table("table_name")` (preferred)
* Aligns with the open-source Apache Spark Declarative Pipelines standard
* Adds `@dp.materialized_view()` as a dedicated decorator (not available in dlt)

### How they connect

```
 sdp_pipeline_config.json  ◄── read by ──►  sdp_customer_pipeline.py (dlt)
        │                                    sdp_customer_pipeline_dp.py (dp)
        ▼
 ┌── customer ─────────────┐    ┌── orders ───────────────┐
 │ customer_bronze          │    │ orders_bronze            │
 │       │                  │    │       │                  │
 │       ▼                  │    │       ▼                  │
 │ customer_silver          │    │ orders_silver            │
 └─────────────────────────┘    └──────────────────────────┘
```

Use **either** Python file as the pipeline's source code (not both — they define the same tables). See **Section 10** for a detailed comparison of `@dp` vs `@dlt`.

## 8. Creating & Running a Pipeline

### Step 1 — Create the pipeline in the UI
1. Go to **Data Engineering → Pipelines → Create pipeline**.
2. Choose **Lakeflow Spark Declarative Pipeline** (not Managed Ingestion).
3. Set a name, e.g. `customer_bronze_silver`.

### Step 2 — Add source files
Under **Source code**, click **Add source code** and add **one** of the pipeline files:
* `/Repos/training/dox_training/sdp_customer_pipeline.py` (classic `@dlt` API), **or**
* `/Repos/training/dox_training/sdp_customer_pipeline_dp.py` (modern `@dp` API)

> ⚠️ Do **not** add both — they define the same tables and would conflict.

### Step 3 — Set destination
* **Catalog**: `one_dbx_training`
* **Schema**: `default`

### Step 4 — Configure compute
* **Cluster mode**: choose **Serverless** for lowest cost.
* **Channel**: use `Current` (latest stable runtime).

### Step 5 — Run the pipeline
* Click **Start → Full refresh** for the first run (loads all CSV files).
* Subsequent runs use **Start** (incremental — only new files).

### Trigger modes

| Mode | Behaviour |
|---|---|
| **Triggered** | Runs once, processes all pending data, then stops |
| **Continuous** | Keeps running, processing new data as it arrives |

> 💡 Use **Triggered** for batch-style pipelines. Use **Continuous** for low-latency streaming.

## 9. Monitoring a Pipeline

### Pipeline graph
The Pipeline UI shows a visual DAG of your tables. Each node is colour-coded:

| Colour | Meaning |
|---|---|
| Green | Running / succeeded |
| Red | Failed |
| Yellow | Waiting for upstream |
| Grey | Not started |

### Per-table details
Click any table node to see:
* **Data quality** — pass/fail counts per expectation
* **Schema** — column names and types
* **Row counts** — rows written in the last update

### Event log
The event log records every pipeline action:
```sql
SELECT timestamp, event_type, message, details
FROM event_log(TABLE(one_dbx_training.default.customer_bronze))
ORDER BY timestamp DESC
LIMIT 50;
```

### Key events to watch

| Event type | What it means |
|---|---|
| `update_progress` | A table update started or completed |
| `flow_progress` | A streaming flow (Auto Loader) made progress |
| `data_quality` | An expectation check result with pass/fail counts |
| `create_update` | A full refresh or new pipeline run started |

## 10. @dp vs @dlt — API Comparison

The Databricks SDP framework offers two Python APIs. Both produce the same underlying pipeline; the difference is **syntax and alignment with the open-source standard**.

### Why two APIs exist

| | `import dlt` (classic) | `from pyspark import pipelines as dp` (modern) |
|---|---|---|
| **Origin** | Databricks-proprietary, shipped since DLT was introduced | Apache Spark open-source standard (DBR 16.0+) |
| **Status** | Fully supported; not deprecated | Preferred for all new code |
| **Ecosystem** | All existing documentation, blog posts, community examples | Growing; official Databricks docs use it for new content |

### Decorator mapping

| Purpose | `@dlt` (classic) | `@dp` (modern) |
|---|---|---|
| Streaming table | `@dlt.table()` + `readStream` | `@dp.table()` + `readStream` |
| Materialized view | `@dlt.table()` + batch `read` | `@dp.materialized_view()` ← **dedicated decorator** |
| Temporary view | `@dlt.view()` | `@dp.temporary_view()` |
| Expect (warn) | `@dlt.expect(name, expr)` | `@dp.expect(name, expr)` |
| Expect or drop | `@dlt.expect_or_drop(name, expr)` | `@dp.expect_or_drop(name, expr)` |
| Expect or fail | `@dlt.expect_or_fail(name, expr)` | `@dp.expect_or_fail(name, expr)` |
| Multiple expects | `@dlt.expect_all(rules)` | `@dp.expect_all(rules)` |
| Create streaming table | `dlt.create_streaming_table()` | `dp.create_streaming_table()` |
| Auto CDC | `dlt.apply_changes()` (legacy) | `dp.create_auto_cdc_flow()` |
| Sink | Not available | `dp.create_sink()` |

### Reading from other pipeline tables

This is the **most important practical difference**:

```python
# ❌ Classic dlt — uses a custom read API (deprecated)
import dlt
df = dlt.read_stream("customer_bronze")      # dlt-proprietary reader
df = dlt.read("customer_bronze")             # dlt-proprietary reader

# ✅ Modern dp — uses standard Spark APIs (preferred)
from pyspark import pipelines as dp
df = spark.readStream.table("customer_bronze")   # standard Spark reader
df = spark.read.table("customer_bronze")         # standard Spark reader
```

Using standard `spark.read.table()` / `spark.readStream.table()` means your code is **portable** — it works identically inside and outside a pipeline context.

### Side-by-side: silver table example

**Classic `@dlt`:**
```python
import dlt
from pyspark.sql.functions import upper, trim, col

@dlt.expect_or_drop("valid_key", "c_custkey IS NOT NULL")
@dlt.table(name="customer_silver", comment="Cleaned customer data")
def customer_silver():
    return (
        dlt.read_stream("customer_bronze")
        .withColumn("c_mktsegment", upper(trim(col("c_mktsegment"))))
    )
```

**Modern `@dp`:**
```python
from pyspark import pipelines as dp
from pyspark.sql.functions import upper, trim, col

@dp.expect_or_drop("valid_key", "c_custkey IS NOT NULL")
@dp.table(name="customer_silver", comment="Cleaned customer data")
def customer_silver():
    return (
        spark.readStream.table("customer_bronze")
        .withColumn("c_mktsegment", upper(trim(col("c_mktsegment"))))
    )
```

### When to use which

| Scenario | Recommendation |
|---|---|
| **New pipeline from scratch** | Use `@dp` — it is the forward-looking standard |
| **Existing pipeline already uses `@dlt`** | Keep `@dlt` — no need to rewrite working code |
| **Migrating to open-source Spark** | Use `@dp` — code is portable without changes |
| **Materialized view needs a dedicated decorator** | Use `@dp.materialized_view()` for clarity |
| **Need sinks (external Delta, Kafka writes)** | Use `@dp` — `dp.create_sink()` has no dlt equivalent |
| **Following older tutorials / blog posts** | `@dlt` examples are more common in legacy content |

> 💡 **Bottom line:** For new work, always prefer `@dp`. The `@dlt` API is fully supported but will receive no new features. Both are functionally equivalent for streaming tables and expectations — only `@dp` has `materialized_view()` and `create_sink()`.